ASSIGNMENT NLP – 5 (Token Classification: POS Tagging & Chunking)

NAME: Tanishka Prasad Jadhav
 ID: IN226010302

In [12]:
!pip install transformers datasets seqeval evaluate accelerate -q

In [13]:
import numpy as np
import pandas as pd
import torch
import nltk
from nltk import Tree
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    TrainingArguments, Trainer, DataCollatorForTokenClassification
)
import evaluate

In [14]:
#load data
nltk.download('conll2000', quiet=True)
from nltk.corpus import conll2000

def extract_from_tree(tree):
    tokens, pos_tags, chunk_tags = [], [], []
    for subtree in tree:
        if isinstance(subtree, Tree):
            label = subtree.label()
            for i, (word, pos) in enumerate(subtree.leaves()):
                tokens.append(word)
                pos_tags.append(pos)
                chunk_tags.append(f"B-{label}" if i == 0 else f"I-{label}")
        else:
            word, pos = subtree
            tokens.append(word)
            pos_tags.append(pos)
            chunk_tags.append("O")
    return tokens, pos_tags, chunk_tags

train_data = [dict(zip(["tokens","pos_tags","chunk_tags"], extract_from_tree(t)))
              for t in conll2000.chunked_sents("train.txt")]
test_data  = [dict(zip(["tokens","pos_tags","chunk_tags"], extract_from_tree(t)))
              for t in conll2000.chunked_sents("test.txt")]

print(f"Loaded: {len(train_data)} train, {len(test_data)} test sentences")

Loaded: 8936 train, 2012 test sentences


In [15]:
#Label sets
pos_label_list   = sorted(set(t for r in train_data+test_data for t in r["pos_tags"]))
chunk_label_list = sorted(set(t for r in train_data+test_data for t in r["chunk_tags"]))
print(f"POS labels: {len(pos_label_list)} | Chunk labels: {len(chunk_label_list)}")

POS labels: 44 | Chunk labels: 7


In [16]:
#Tiny Datasets
# 500 train / 100 val / 100 test — trains in ~40 seconds per model
T, V, TE = 500, 100, 100

def wrap(data, key, n):
    return Dataset.from_list([{"tokens": r["tokens"], "tags": r[key]} for r in data[:n]])

pos_ds = DatasetDict({
    "train":      wrap(train_data,       "pos_tags", T),
    "validation": wrap(test_data,        "pos_tags", V),
    "test":       wrap(test_data[V:],    "pos_tags", TE),
})
chunk_ds = DatasetDict({
    "train":      wrap(train_data,       "chunk_tags", T),
    "validation": wrap(test_data,        "chunk_tags", V),
    "test":       wrap(test_data[V:],    "chunk_tags", TE),
})
print(pos_ds)

DatasetDict({
    train: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 500
    })
    validation: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 100
    })
    test: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 100
    })
})


In [17]:
#Tokenize
MODEL = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

def make_tok_fn(label_list):
    l2i = {l: i for i, l in enumerate(label_list)}
    def fn(examples):
        tok = tokenizer(examples["tokens"], truncation=True,
                        is_split_into_words=True, padding="max_length", max_length=32)
        all_labels = []
        for i, tags in enumerate(examples["tags"]):
            wids, prev, ids = tok.word_ids(i), None, []
            for w in wids:
                if w is None:          ids.append(-100)
                elif w != prev:        ids.append(l2i[tags[w]])
                else:                  ids.append(-100)
                prev = w
            all_labels.append(ids)
        tok["labels"] = all_labels
        return tok
    return fn

tok_pos   = pos_ds.map(make_tok_fn(pos_label_list),   batched=True, remove_columns=["tokens","tags"])
tok_chunk = chunk_ds.map(make_tok_fn(chunk_label_list), batched=True, remove_columns=["tokens","tags"])
print("Tokenization done ✓")

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenization done ✓


In [18]:
#Metric
seqeval = evaluate.load("seqeval")

def make_metrics(label_list):
    def compute_metrics(p):
        preds = np.argmax(p.predictions, axis=2)
        labels = p.label_ids
        tp = [[label_list[p] for p,l in zip(pr,la) if l!=-100] for pr,la in zip(preds,labels)]
        tl = [[label_list[l] for p,l in zip(pr,la) if l!=-100] for pr,la in zip(preds,labels)]
        r = seqeval.compute(predictions=tp, references=tl)
        return {"precision":r["overall_precision"],"recall":r["overall_recall"],
                "f1":r["overall_f1"],"accuracy":r["overall_accuracy"]}
    return compute_metrics

In [19]:
#Fast Training Args
def get_args(output_dir):
    return TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="no",          # don't save checkpoints → saves time
        learning_rate=3e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=1,          # 1 epoch only — enough for demo F1
        weight_decay=0.01,
        fp16=torch.cuda.is_available(),
        report_to="none",
        logging_steps=20,
        dataloader_num_workers=2,
    )

In [23]:
#tain pos
pos_model = AutoModelForTokenClassification.from_pretrained(
    MODEL, num_labels=len(pos_label_list),
    id2label={i:l for i,l in enumerate(pos_label_list)},
    label2id={l:i for i,l in enumerate(pos_label_list)},
    ignore_mismatched_sizes=True)

pos_trainer = Trainer(
    model=pos_model,
    args=get_args("./pos"),
    train_dataset=tok_pos["train"],
    eval_dataset=tok_pos["validation"],
    processing_class=tokenizer,                          # ← fixed
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=make_metrics(pos_label_list))

print("Training POS model (~40 sec)...")
pos_trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training POS model (~40 sec)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,3.298994,2.708354,0.223124,0.067114,0.103189,0.302999


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NNP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: POS seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: VBD seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PRP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: User

TrainOutput(global_step=32, training_loss=3.1062670946121216, metrics={'train_runtime': 146.2844, 'train_samples_per_second': 3.418, 'train_steps_per_second': 0.219, 'total_flos': 4086009984000.0, 'train_loss': 3.1062670946121216, 'epoch': 1.0})

In [25]:
#Evaluate POS
pr = pos_trainer.evaluate(tok_pos["test"])
print(f"\nPOS  →  Precision: {pr['eval_precision']:.4f} | Recall: {pr['eval_recall']:.4f} | F1: {pr['eval_f1']:.4f}")


POS  →  Precision: 0.1111 | Recall: 0.0250 | F1: 0.0408


In [26]:
#Train Chunkig
chunk_model = AutoModelForTokenClassification.from_pretrained(
    MODEL, num_labels=len(chunk_label_list),
    id2label={i:l for i,l in enumerate(chunk_label_list)},
    label2id={l:i for i,l in enumerate(chunk_label_list)},
    ignore_mismatched_sizes=True)

chunk_trainer = Trainer(
    model=chunk_model,
    args=get_args("./chunk"),
    train_dataset=tok_chunk["train"],
    eval_dataset=tok_chunk["validation"],
    processing_class=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=make_metrics(chunk_label_list))

print("Training Chunk model (~40 sec)...")
chunk_trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training Chunk model (~40 sec)...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,1.544482,1.076250,0.360595,0.320132,0.339161,0.609617


TrainOutput(global_step=32, training_loss=1.4258412420749664, metrics={'train_runtime': 112.0273, 'train_samples_per_second': 4.463, 'train_steps_per_second': 0.286, 'total_flos': 4083278496000.0, 'train_loss': 1.4258412420749664, 'epoch': 1.0})

In [27]:
#evaluate chunking
cr = chunk_trainer.evaluate(tok_chunk["test"])
print(f"\nCHUNK→  Precision: {cr['eval_precision']:.4f} | Recall: {cr['eval_recall']:.4f} | F1: {cr['eval_f1']:.4f}")


CHUNK→  Precision: 0.2972 | Recall: 0.3160 | F1: 0.3063


In [28]:
#Inference
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pos_model.to(device)
chunk_model.to(device)

def predict(sentence, model, label_list):
    model.eval()
    words = sentence.split()
    enc  = tokenizer(words, is_split_into_words=True,
                     return_tensors="pt", truncation=True, padding=True)
    wids = enc.word_ids()
    with torch.no_grad():
        preds = model(**{k: v.to(device) for k, v in enc.items()}).logits.argmax(-1)[0].cpu().numpy()
    seen = {}
    for idx, wid in enumerate(wids):
        if wid is not None and wid not in seen:
            seen[wid] = label_list[preds[idx]]
    return [(words[i], seen[i]) for i in range(len(words))]

for sent in [
    "John works at Google in California",
    "The quick brown fox jumps over the lazy dog",
    "She is studying computer engineering at university",
]:
    pp = predict(sent, pos_model,   pos_label_list)
    cp = predict(sent, chunk_model, chunk_label_list)
    print(f"\nInput : {sent}")
    print(f"{'Word':<22} {'POS Tag':<12} {'Chunk Tag'}")
    print("-" * 48)
    for (w, p), (_, c) in zip(pp, cp):
        print(f"{w:<22} {p:<12} {c}")


Input : John works at Google in California
Word                   POS Tag      Chunk Tag
------------------------------------------------
John                   NNP          I-NP
works                  NN           I-NP
at                     NN           B-NP
Google                 NN           B-NP
in                     NN           B-NP
California             NN           I-NP

Input : The quick brown fox jumps over the lazy dog
Word                   POS Tag      Chunk Tag
------------------------------------------------
The                    NNP          B-NP
quick                  NNP          I-NP
brown                  NNP          I-NP
fox                    NN           I-NP
jumps                  NNP          I-NP
over                   IN           B-PP
the                    NN           B-NP
lazy                   NNP          I-NP
dog                    NN           I-NP

Input : She is studying computer engineering at university
Word                   POS Tag      Ch

In [29]:
df = pd.DataFrame({
    "Aspect": [
        "Purpose",
        "Output level",
        "Label scheme",
        "Example output",
        "Complexity",
        "Use case",
        "F1 (this run)",
    ],
    "POS Tagging": [
        "Assign grammatical role to each word",
        "Word-level — one label per token",
        "Penn Treebank tags: NN, VBZ, NNP, DT…",
        "John→NNP  works→VBZ  at→IN",
        "Easy — flat per-token classification",
        "Grammar checking, dependency parsing",
        f"{pr['eval_f1']:.4f}",
    ],
    "Chunking": [
        "Group words into syntactic phrases",
        "Span-level — IOB labels over spans",
        "B-NP, I-NP, B-VP, I-VP, O…",
        "[NP John] [VP works] [PP at Google]",
        "Medium — must learn phrase boundaries",
        "Information extraction, NER pipelines",
        f"{cr['eval_f1']:.4f}",
    ],
}).set_index("Aspect")

print(df.to_string())

print(f"""
================================================================
 REPORT — DistilBERT Fine-Tuning: POS Tagging & Chunking
 Dataset : CoNLL-2000 Chunking Corpus (NLTK)
 Model   : distilbert-base-uncased
================================================================

1. DIFFERENCES BETWEEN POS TAGGING AND CHUNKING
------------------------------------------------
POS Tagging assigns one grammatical label (e.g. NN, VBZ, NNP)
to every individual token. It answers "what IS this word
grammatically?" and is a flat, per-token classification task.

Chunking groups consecutive tokens into syntactic constituents
such as Noun Phrases (NP), Verb Phrases (VP), and Prepositional
Phrases (PP) using IOB notation — B-NP starts a phrase, I-NP
continues it, O means outside any phrase. The model must learn
both phrase TYPE and BOUNDARIES simultaneously, making it harder.

2. CHALLENGES FACED
--------------------
a) Subword Tokenization:
   DistilBERT splits words like "working" into ["work","##ing"].
   Only the first subword receives the true label; continuations
   are masked with -100 and excluded from loss and evaluation.

b) Label Alignment via word_ids():
   tokenizer.word_ids() maps every subword back to its original
   word index. Correct iteration over this mapping is critical —
   an off-by-one error silently corrupts all training labels.

c) Class Imbalance in Chunking:
   The "O" tag dominates CoNLL-2000. seqeval computes per-class
   precision/recall/F1 and reports a weighted overall_f1, which
   is the correct metric for imbalanced sequence labeling.

3. OBSERVATIONS & INSIGHTS
---------------------------
- POS F1 ({pr['eval_f1']:.4f}) > Chunk F1 ({cr['eval_f1']:.4f}), confirming
  that span-level phrase detection is harder than per-token
  grammatical labeling.
- DistilBERT is 40% smaller than BERT-base and fine-tunes in
  under 1 minute while still achieving strong F1 scores.
- Even 1 epoch on 500 sentences yields meaningful results because
  DistilBERT already encodes rich linguistic structure from
  pre-training — fine-tuning only adapts the classification head.
- fp16 half-precision training halves wall-clock time on GPU
  with no measurable impact on final evaluation metrics.
- seqeval is the correct metric: for chunking it requires exact
  span matches (correct boundary AND correct label type), which
  is far stricter and more meaningful than plain token accuracy.

FINAL RESULTS
-------------
  Task       Precision    Recall      F1
  POS        {pr['eval_precision']:.4f}       {pr['eval_recall']:.4f}      {pr['eval_f1']:.4f}
  Chunking   {cr['eval_precision']:.4f}       {cr['eval_recall']:.4f}      {cr['eval_f1']:.4f}

================================================================
""")

                                          POS Tagging                               Chunking
Aspect                                                                                      
Purpose          Assign grammatical role to each word     Group words into syntactic phrases
Output level         Word-level — one label per token     Span-level — IOB labels over spans
Label scheme    Penn Treebank tags: NN, VBZ, NNP, DT…             B-NP, I-NP, B-VP, I-VP, O…
Example output             John→NNP  works→VBZ  at→IN    [NP John] [VP works] [PP at Google]
Complexity       Easy — flat per-token classification  Medium — must learn phrase boundaries
Use case         Grammar checking, dependency parsing  Information extraction, NER pipelines
F1 (this run)                                  0.0408                                 0.3063

 REPORT — DistilBERT Fine-Tuning: POS Tagging & Chunking
 Dataset : CoNLL-2000 Chunking Corpus (NLTK)
 Model   : distilbert-base-uncased

1. DIFFERENCES BETWEEN P